# HOUSE-3 · 3 — In the field

**The failures this notebook prevents** are all the same failure: a readout that looks
like the analysis the plan committed to and is not. The dashboard that was open every
morning and called the search test in its second week. The display test that stopped
early and reported the number it stopped on. The doubled video budget the platform only
two-thirds delivered, read as if it had been doubled. The search holdout whose control
markets went dark, read on whoever was left. The assignment somebody re-drew after the
plan was frozen and never wrote down.

Each of those produces a number with an interval, and each of the numbers is wrong in a
way its interval does not show. This notebook runs the year and reads every experiment
the way the plan said it would be read — and where the field departed from the plan,
records the departure on the run, so that notebook 4 can tell which readouts are
evidence.

| | reaches into |
|---|---|
| the dashboard, priced | `design.confidence_sequence`, `design.AnytimeLook`, `design.evalue` |
| scheduled looks and the number a stopped study may report | `design.LookSchedule`, `design.obrien_fleming`, `design.StoppingRule`, `design.monitor`, `build.ExperimentBuilder.read_stopped`, `design.StoppedEstimate` |
| did it happen as assigned | `diagnose.check_delivery`, `diagnose.DeliveryReport`, `identify.response_to_dose`, `identify.compliance` |
| reached is not observed | `diagnose.attrition`, `identify.lee_bounds` |
| the neighbour or the effect | `design.factorial`, `design.CONCURRENT_EXPERIMENTS` |
| reading and filing | `design.estimate`, `design.panel_arrays`, `calibrate.Measurement`, `build.ExperimentBuilder.read`, `io.ExperimentRun.deviate`, `io.ExperimentRun.conformance` |

In [ ]:
import sys

sys.path.insert(0, ".")

import numpy as np
import pandas as pd
from IPython.display import display

import house as H
import year as Y
from axiom.display import enable, table
from _style import (
    AQUA, BLUE, CRITICAL, GOOD, MUTED, ORANGE, VIOLET, WARNING,
    annotate, caption, compare, curve_band, density, dumbbell, figure, heat, intervals,
    lines, mark_x, mark_y, points, shade, steps,
)

enable();  # every axiom result renders itself from here on

from scipy import stats as st

worlds = {name: H.world(name, year=1) for name in H.PARTIES}
last_year = {name: H.world(name, year=0) for name in H.PARTIES}
inputs = {name: Y.design_inputs(w) for name, w in last_year.items()}
catalog, programs, builders = Y.open_house()
Y.register_definitions(builders)
plans = Y.plan_year(builders, inputs)          # notebook 2, replayed
fielded = Y.field_year(worlds, plans)          # the year, as the world ran it
runs = {k: builders[p.experiment.party].start(p.run, at=f"2026-W{p.experiment.start:02d}") for k, p in plans.items()}

summary = Y.fielded_frame(worlds, fielded)
display(summary.round(1))
print("the last column is the world's, not the house's: the true lift per treated market-week, which no notebook uses to decide anything")

## 1 · The dashboard, priced

Northwind's search holdout had a dashboard. Every Monday the analyst read the
difference-in-differences so far, and every Monday the interval was a 95% interval —
for one look. Read eight times, a fixed-sample interval excludes the truth about a fifth
of the time (`nbs/design/07`), and the first Monday's reading below — a p-value a hair
under 0.05 on one week of data — is exactly the kind that gets a test called early.

`confidence_sequence` takes the same statistics and gives an interval valid at *every*
look at once. The price is width, and the e-value it carries is what the programme's
error control in notebook 4 will run on.

In [ ]:
from axiom.design import confidence_sequence

exp = plans["NW-01"].experiment
f = fielded["NW-01"]
kpi = Y.kpi_seen(worlds["northwind"], f)
z, lift, se = Y.weekly_path(kpi, f, range(1, exp.weeks + 1))
fractions = [k / exp.weeks for k in range(1, exp.weeks + 1)]
seq = confidence_sequence(z, fractions, alpha=0.05, name="NW-01 dashboard",
                          labels=[f"week {exp.start + k}" for k in range(exp.weeks)])
rows = []
for look, (l, lift_k, se_k) in enumerate(zip(seq.looks, lift, se)):
    p_fixed = float(2 * st.norm.sf(abs(l.z)))
    rows.append([l.label, f"{lift_k:+.1f} ± {se_k:.1f}", f"{l.z:+.2f}", f"{p_fixed:.3f}", "yes" if p_fixed < 0.05 else "no",
                 l.interval.text(), "yes" if l.crossed else "no", f"{l.evalue:.2f}"])
table(rows, headers=("look", "lift so far", "z", "fixed-sample p", "dashboard says lift", "anytime interval (drift)", "sequence says lift", "e-value"),
      title="eight Mondays on the NW-01 dashboard")
print(seq.summary())

In [ ]:
from _style import band

weeks = [exp.start + k for k in range(exp.weeks)]
se_full = se[-1]
drift = [l.drift for l in seq.looks]
fixed_lo = [l.drift - 1.96 / np.sqrt(l.information) for l in seq.looks]
fixed_hi = [l.drift + 1.96 / np.sqrt(l.information) for l in seq.looks]
fig = figure(title="What the dashboard showed, and what it was entitled to show",
             subtitle="the running drift with its fixed-sample interval (orange) and its anytime interval (blue)",
             x_title="week", y_title="drift (lift / se at full information)")
band(fig, weeks, fixed_lo, fixed_hi, color=ORANGE, name="fixed-sample 95%, read every week")
band(fig, weeks, [l.interval.lower for l in seq.looks], [l.interval.upper for l in seq.looks], color=BLUE, name="95% confidence sequence")
fig.add_scatter(x=weeks, y=drift, mode="lines+markers", line={"color": BLUE, "width": 2}, marker={"size": 8}, name="running estimate")
mark_y(fig, 0.0, text="no lift")
caption(fig, "The orange band clears zero on every one of the eight Mondays, narrowly, and the analyst would have called it on "
             "the first. The blue band never does, and it is the one that is true at every Monday together. Both are correct "
             "answers to different questions; only one is the question a dashboard reader is entitled to ask.")

The experiment ran its length. The readout is the difference-in-differences over the
whole window, filed against the estimand the plan committed to, and the conformance is
`identified`: nothing moved between the plan and the reading.

In [ ]:
from axiom.calibrate import Measurement

e = Y.did(kpi, f)
m = Y.lift_measurement(exp, plans["NW-01"].estimand, e, source="NW-01")
runs["NW-01"] = builders["northwind"].read(runs["NW-01"], m, at="2026-W16")
print(f"NW-01 lift {m.estimate:+.2f} ± {m.se:.2f} conversions per market-week ({e.method}, {e.n_treated} treated, {e.n_control} control)")
print(runs["NW-01"].summary())

## 2 · Four scheduled looks, and the number a stopped study may report

Globex's display test was different: four looks were *scheduled*, at weeks 32, 34, 36
and 38, against an O'Brien–Fleming boundary that spends almost nothing early. It
crossed at the third look. The estimate at a stopping look is biased away from zero —
the study stopped *because* the number was large — and `read_stopped` does the two
things that belong together: it files the median-unbiased estimate as the readout, and
it records a `stopped_early` deviation naming the naive number it replaced.

In [ ]:
from axiom.design import monitor

exp = plans["GX-02"].experiment
f = fielded["GX-02"]
kpi = Y.kpi_seen(worlds["globex"], f)
z, lift, se = Y.weekly_path(kpi, f, (2, 4, 6, 8))
rule = Y.gx02_rule()
path = monitor(rule, z[:3], effects=lift[:3], ses=se[:3])
table([[label, f"{info:.2f}", f"{zk:+.2f}", f"{bound:.2f}", "crossed" if zk > bound else "continue"]
       for label, info, zk, bound in zip(rule.looks.labels[:3], rule.looks.information[:3], z[:3], rule.boundaries[0].z[:3])],
      headers=("look", "information", "z", "boundary", "decision"), title=f"GX-02 against {rule.name}")
print("decision:", path.decision, "at look", path.stopped_at + 1)

runs["GX-02"], corrected = builders["globex"].read_stopped(runs["GX-02"], path, plans["GX-02"].estimand, at="2026-W36")
table([["naive, at the stopping look", f"{corrected.naive_effect:.1f}", "—"],
       ["median-unbiased", f"{corrected.effect:.1f}", corrected.effect_interval.text()]],
      headers=("estimate", "lift per market-week", "interval"), title=f"{corrected.rule}: moved by {corrected.effect - corrected.naive_effect:+.1f}")
print(runs["GX-02"].summary())
print("deviation:", runs["GX-02"].deviations[0].role, "—", runs["GX-02"].deviations[0].reason)

In [ ]:
looks = list(rule.looks.information)
fig = figure(title="Where GX-02 stopped", subtitle="the O'Brien–Fleming boundary and the statistic at each scheduled look",
             x_title="information fraction", y_title="z (positive: display works)")
fig.add_scatter(x=looks, y=list(rule.boundaries[0].z), mode="lines+markers", line={"color": MUTED, "width": 2, "dash": "dot"}, name="efficacy boundary")
fig.add_scatter(x=looks[:3], y=z[:3], mode="lines+markers", line={"color": AQUA, "width": 2}, marker={"size": 9}, name="observed z")
annotate(fig, looks[2], z[2], "stopped here")
mark_y(fig, 1.96, text="fixed-sample 1.96")
caption(fig, "The boundary starts far above 1.96 and comes down to meet it. A statistic that looks 'significant' at the first look is "
             "not, by this rule; at the third it is, and the correction for having stopped there is small — "
             f"{corrected.effect - corrected.naive_effect:+.1f} conversions per market-week — because almost none of the error budget was spent earlier.")

## 3 · Did it happen as assigned?

Before any model, one question: did the markets arrive in the arms the rule put them
in, and did the treatment reach them? `check_delivery` runs every check its arguments
support and names the ones that ran. For the holdouts the split is exact by
construction — permuted blocks — and the sample-ratio check is a formality worth
running. For the dose test it is not a formality: the platform cleared the doubled
budget nowhere in full.

In [ ]:
from axiom.diagnose import check_delivery

exp = plans["GX-01"].experiment
f = fielded["GX-01"]
clean = check_delivery({"control": len(f.control), "treated": len(f.treated)}, Y.ALLOCATIONS[exp.n_treated])
print("GX-01:", clean.summary())

exp = plans["NW-02"].experiment
f = fielded["NW-02"]
report = check_delivery({"control": len(f.control), "treated": len(f.treated)}, Y.ALLOCATIONS[exp.n_treated], exposed=f.exposed())
print("\nNW-02:", report.summary())
print("  checks run:", report.checks_run, "| failures:", report.failures, "| verdict:", report.verdict().status)
table([[market, f"{share:.0%}"] for market, share in sorted(f.delivered_fraction.items())],
      headers=("treated market", "of the assigned increase, delivered"), title="what the platform actually cleared")

### A dose is not a switch

A market that got 70% of a doubling is not a non-complier; it is a market that got a
dose. `response_to_dose` reports the two estimands such an experiment supports: the
intention-to-treat effect of *assigning* the doubling, over every market, and the
local average derivative — conversions per extra kUSD, over the markets the assignment
moved, weighted by how far it moved them. The plan committed to the first. The
departure from "doubled" is recorded on the run as a deviation, asserted: the readout is
still the effect of the assignment northwind can actually make.

In [ ]:
from axiom.identify import response_to_dose

kpi = Y.kpi_seen(worlds["northwind"], f)
change = kpi[:, exp.post].mean(axis=1) - kpi[:, exp.pre].mean(axis=1)
extra = (f.doses["video"][:, exp.post] - worlds["northwind"].doses["video"][:, exp.post]).mean(axis=1)
frame = pd.DataFrame({"market": H.MARKETS, "change": change,
                      "assigned": [1.0 if m_ in f.treated else 0.0 for m_ in H.MARKETS], "extra_spend": extra})
dose = response_to_dose(frame, "change", "assigned", "extra_spend")
print(dose.summary())
print("verdict:", dose.verdict().status)

e = Y.did(kpi, f)
m = Y.lift_measurement(exp, plans["NW-02"].estimand, e, source="NW-02")
run = runs["NW-02"]
for line in report.ledger():
    run = run.with_ledger_line(line)
run = run.deviate("dose", f"{1.0 + exp.uplift * float(np.mean(list(f.delivered_fraction.values()))):.2f}x delivered on average",
                  reason="the platform cleared 65-85% of the doubled budget; the readout is the effect of assigning the doubling, not of receiving it",
                  state="asserted", at="2026-W24")
runs["NW-02"] = builders["northwind"].read(run, m, at="2026-W28")
print(f"\nfiled: ITT lift {m.estimate:+.2f} ± {m.se:.2f} | conformance {runs['NW-02'].conformance().status}")

### Assigned is not received

Globex's search holdout could not switch search off everywhere: three of the ten
treated markets carry retailer co-op search the house does not control. `compliance`
reports the intention-to-treat effect and the complier effect side by side, with the
population each is over and the assumptions that license the second. The plan
committed to the first; the complier effect goes into notebook 5's corpus as its own
record, where the pool will refuse to average it with the others.

In [ ]:
from axiom.identify import compliance

exp = plans["GX-03"].experiment
f = fielded["GX-03"]
kpi = Y.kpi_seen(worlds["globex"], f)
change = kpi[:, exp.post].mean(axis=1) - kpi[:, exp.pre].mean(axis=1)
frame = pd.DataFrame({"change": change,
                      "assigned": [1.0 if m_ in f.treated else 0.0 for m_ in H.MARKETS],
                      "off": [1.0 if (m_ in f.treated and f.delivered_fraction[m_] > 0) else 0.0 for m_ in H.MARKETS]})
comp = compliance(frame, "change", "assigned", "off")
print(comp.summary())
print("verdict:", comp.verdict().status, "| complier share", f"{comp.share:.0%}")

e = Y.did(kpi, f)
m = Y.lift_measurement(exp, plans["GX-03"].estimand, e, source="GX-03")
run = runs["GX-03"]
for line in comp.ledger():
    run = run.with_ledger_line(line)
runs["GX-03"] = builders["globex"].read(run, m, at="2026-W48")
complier_lift = -comp.complier.estimate
complier_se = comp.complier.se
print(f"\nfiled: ITT lift {m.estimate:+.1f} ± {m.se:.1f}; the complier effect {complier_lift:+.1f} ± {complier_se:.1f} "
      f"is carried on the run's ledger, not as the readout | conformance {runs['GX-03'].conformance().status}")

## 4 · Reached is not observed

In acme's autumn search holdout the KPI feed vendor dropped ten control markets after
the second week — the ones with a week too quiet to bill for — and nobody re-enabled
them, because nobody was watching the control arm. `attrition` asks whether the outcome
got back at one rate in both arms. It did not, and the verdict is `blocked`: no
assumption over observables licenses a contrast between two arms whose reporters are
different populations.

What survives is a **bound**, over a different population — the markets that would have
reported either way — and that is `lee_bounds`.

In [ ]:
from axiom.diagnose import attrition
from axiom.identify import lee_bounds

exp = plans["AC-04"].experiment
f = fielded["AC-04"]
kpi = Y.kpi_seen(worlds["acme"], f)
observed = H.observed_markets(kpi, exp.post)
counts = {"control": len(f.control), "treated": len(f.treated)}
reported = {"control": sum(1 for i in observed if H.MARKETS[i] in f.control),
            "treated": sum(1 for i in observed if H.MARKETS[i] in f.treated)}
att = attrition(counts, reported, alpha=0.01)
print(att.summary())
print("  (alpha 0.01 rather than the weekly-check 0.001: this check runs once per readout, nine times a year)")

change = np.nanmean(kpi[:, exp.post], axis=1) - kpi[:, exp.pre].mean(axis=1)
frame = pd.DataFrame({"change": change, "assigned": [1.0 if m_ in f.treated else 0.0 for m_ in H.MARKETS],
                      "reported": [1.0 if i in observed else 0.0 for i in range(H.N_MARKETS)]})
bounds = lee_bounds(frame, "change", "assigned", "reported")
truth = float(summary.set_index("experiment").loc["AC-04", "true_lift"])
e = Y.did(kpi, f)
table([["naive, on the markets that reported", f"{-e.effect:+.1f} ± {e.se:.1f}", "all assigned markets (claimed)"],
       ["Lee bounds", f"[{-bounds.upper:+.1f}, {-bounds.lower:+.1f}]", f"markets that report either way; {bounds.trimmed:.0%} of the treated arm trimmed to match"],
       ["the world's truth", f"{truth:+.1f}", "all assigned markets"]],
      headers=("", "lift per market-week", "population"), title="AC-04: what can be said, and about whom")

In [ ]:
fig = intervals([("naive, reporters only", -e.effect, -e.effect - 1.96 * e.se, -e.effect + 1.96 * e.se),
                 ("Lee bounds (always-reporters)", -(bounds.upper + bounds.lower) / 2, -bounds.upper, -bounds.lower)],
                ref=truth, ref_label="the truth", highlight="Lee bounds (always-reporters)",
                title="A number over the wrong population, and a bound over the right one",
                subtitle="AC-04 search lift per market-week", x_title="conversions per market-week")
caption(fig, "The control markets that went dark were the ones whose demand fell, so the survivors flatter the channel: the naive "
             f"estimate sits {(-e.effect - truth) / e.se:.1f} standard errors above the truth. The bound is wide because half the control "
             "arm was trimmed to match, and it is a statement about markets that would have reported whatever happened.")

The house files the naive number — it is the only point estimate there is — but not as
the analysis the plan committed to. The deviation says the population moved, its state
is `unverified`, the attrition report's ledger line rides on the run, and the conformance
is `downgraded`. Notebook 4 will decide whether a downgraded readout is evidence, and
notebook 5 will find that its estimand names a different population.

In [ ]:
from axiom.core import Population

m = Y.lift_measurement(exp, plans["AC-04"].estimand, e, source="AC-04")
run = runs["AC-04"].with_ledger_line(att.ledger_line()).deviate(
    "population", Population(name="markets_whose_feed_stayed_up"),
    reason="ten control markets went dark after week 44; the readout is over the thirty that reported",
    state="unverified", at="2026-W50")
runs["AC-04"] = builders["acme"].read(run, m, at="2026-W50")
print(runs["AC-04"].summary())

## 5 · The neighbour or the effect

Acme's two spring tests ran on the same markets for four weeks, moving different levers.
Notebook 2 called that *concurrent*: unbiased in expectation, at a variance cost, under
the assumption that social and search do not interact. `CONCURRENT_EXPERIMENTS` names
the analysis that checks it — a factorial of the two assignments — and now the data
exist. If the interaction were real, neither main effect would be the effect of its own
channel; it would be an average over what the other test happened to be doing.

In [ ]:
from axiom.design import CONCURRENT_EXPERIMENTS, factorial

f1, f2 = fielded["AC-01"], fielded["AC-02"]
shared = slice(max(f1.experiment.start, f2.experiment.start), min(f1.experiment.stop, f2.experiment.stop))
kpi = Y.kpi_seen(worlds["acme"], f2)   # AC-02's doses carry AC-01's holdout too: the year stacks
y = kpi[:, shared].mean(axis=1) - kpi[:, f1.experiment.pre].mean(axis=1)
frame = pd.DataFrame({"y": y, "ac01": [1.0 if m_ in f1.treated else 0.0 for m_ in H.MARKETS],
                      "ac02": [1.0 if m_ in f2.treated else 0.0 for m_ in H.MARKETS]})
check = factorial(frame, "y", "ac01", "ac02")
print(check.summary())
print("\nverdict:", check.verdict().status, "—", check.verdict().reason[:160])
print("power note:", check.power_note)

`unverified`, never `identified`: an interaction is about twice as hard to detect as a
main effect of the same size, so a study powered for mains failing to find one is weak
evidence, and the report says what it *could* have seen. The assumption stays on both
runs' ledgers with its state, and both are read against their committed estimands.

In [ ]:
for key in ("AC-01", "AC-02"):
    f = fielded[key]
    exp = plans[key].experiment
    e = Y.did(Y.kpi_seen(worlds["acme"], f), f)
    m = Y.lift_measurement(exp, plans[key].estimand, e, source=key)
    run = runs[key].with_ledger_line(check.ledger_line())
    runs[key] = builders["acme"].read(run, m, at=f"2026-W{exp.stop:02d}")
    print(f"{key}: lift {m.estimate:+.2f} ± {m.se:.2f} | conformance {runs[key].conformance().status}")

## 6 · The field ran two weeks long

Globex's tv holdout could not be restarted until week 18; the tv buy has a lead time.
The occupancy and the window both moved, and both moves are `deviate`d with a reason
and the state `asserted` — the operator says the ten-week readout is still the quantity
the plan named. The conformance drops to `downgraded` and carries each departure as an
`Assumption` a reader can challenge.

In [ ]:
exp = plans["GX-01"].experiment
f = fielded["GX-01"]                       # fielded over ten weeks, as it ran
kpi = Y.kpi_seen(worlds["globex"], f)
longer = f.experiment
run = runs["GX-01"].deviate(
    "occupancy", H.occupancy(longer, H.MARKETS),
    reason="the tv buy could not be restarted until week 18; the holdout ran ten weeks", state="asserted", at="2026-W16",
).deviate("window", longer.window, reason="read over the ten weeks the holdout ran", state="asserted", at="2026-W18")
e = Y.did(kpi, f, post=longer.post)
m = Y.lift_measurement(exp, plans["GX-01"].estimand, e, source="GX-01")
runs["GX-01"] = builders["globex"].read(run, m, at="2026-W18")
print(runs["GX-01"].summary())
verdict = runs["GX-01"].conformance()
table([[a.name, a.state, a.challenged_by[:90]] for a in verdict.assumptions],
      headers=("assumption", "state", "challenged by"), title="what a reader of GX-01 is being asked to accept")

## 7 · The change nobody filed

Northwind's second search test was moved to the autumn in notebook 2 and committed. In
week 38 the analyst decided the spring assignment "looked unlucky" and drew a new one —
a different seed, a different ten markets — and read the experiment on it. Nothing was
filed. The committed plan's hash no longer matches the roles the run was read under,
there is no deviation to explain why, and the conformance is `blocked`: not
*downgraded*, because a recorded change can be argued with and an unrecorded one
cannot.

In [ ]:
exp = plans["NW-03"].experiment
redrawn = Y.arms(exp, seed=99)
tampered = runs["NW-03"].model_copy(update={"roles": {**runs["NW-03"].roles, "assignment": redrawn.spec.content_hash()}})
f = H.field_experiment(worlds["northwind"], exp, redrawn.units_in("treated"), redrawn.units_in("control"), doses=fielded["NW-03"].doses)
e = Y.did(Y.kpi_seen(worlds["northwind"], f), f)
m = Y.lift_measurement(exp, plans["NW-03"].estimand, e, source="NW-03")
runs["NW-03"] = builders["northwind"].read(tampered, m, at="2026-W48")
print(f"NW-03 as read: lift {m.estimate:+.2f} ± {m.se:.2f} — a perfectly good-looking number")
print(runs["NW-03"].summary())
print("\nplanned assignment", plans["NW-03"].run.roles["assignment"][:12], "| read under", runs["NW-03"].roles["assignment"][:12])

## 8 · The year, as filed

Nine runs, four answers. `identified` where nothing moved; `downgraded` where something
did and was written down, with the departure carried as an assumption; `blocked` where
it was not. The runs are in the catalog under their parties, and `readouts_across` —
the input the programme's error control takes — returns the licensed ones and quietly
leaves NW-03 out, because a readout the run itself calls blocked is not evidence about
anything.

In [ ]:
from axiom.build import readouts_across

rows = []
for key in plans:
    run = runs[key]
    v = run.conformance()
    rows.append([key, run.scope.split("/")[0], run.stage, len(run.deviations), len(run.ledger), v.status, (v.reason or "—")[:70]])
table(rows, headers=("experiment", "party", "stage", "deviations", "ledger lines", "conformance", "why"), title="the year, as filed")
licensed = readouts_across(list(builders.values()))
print("readouts the programme will take:", sorted(r.experiment for r in licensed))
print("filed in the catalog:", len(catalog), "artifacts;", len(catalog.find(type_name="ExperimentRun", tags={"stage": "read"})), "runs at 'read'")

# The replay module reproduces this notebook's filings, so notebooks 4 and 5 can start from them.
replay_catalog, _, replay_builders = Y.open_house()
Y.register_definitions(replay_builders)
replay = Y.read_year(worlds, replay_builders, Y.plan_year(replay_builders, inputs), fielded)
same = all(replay[k].run.readout_hash == runs[k].readout_hash and replay[k].run.conformance().status == runs[k].conformance().status for k in plans)
print("year.read_year reproduces every readout hash and conformance verdict:", same)

## What notebook 3 established

1. **A dashboard read every Monday is not a 95% interval.** The fixed-sample band cleared
   zero on every Monday, narrowly; the confidence sequence never did, and the readout
   carries the e-value the programme will use instead of the Monday that looked best.
2. **A stopped study reports the median-unbiased number**, and files the naive one as a
   deviation, in one call.
3. **The delivery check runs before any model.** The dose test was delivered two-thirds;
   its readout is the effect of the assignment, with the derivative per kUSD beside it
   and the shortfall on the run.
4. **Reached is not observed.** Differential attrition is `blocked`, the naive number
   sits well above the truth, and what survives is a bound over the markets that would
   have reported either way.
5. **The concurrent pair was checked**, and the answer is `unverified`, with the
   interaction the study could have seen printed beside it.
6. **Four verdicts, not a boolean.** Nine runs filed; eight licensed; one blocked for
   an unrecorded change, and left out of every count downstream.